# Game Simulator

In [1]:
# show
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from common_lib.sql import BigQueryConnector
from common_lib.simulation import (
    SimulationEngine, PlatformInputs,
    save_scenario, load_scenario, list_scenarios,
    save_result, load_result, list_results,
)
from common_lib.widgets import ScenarioPanel
from common_lib.tables import monthly_table, comparison_table, export_all_tables, comparison_table
import datetime as dt

In [ ]:
refresh_data = False  # Set to True to refresh data from BigQuery, False to load from local pickle
if refresh_data:
    bqc = BigQueryConnector()

In [3]:
# show

params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}
#cost = bqc.print_cost_estimate('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)

In [4]:
PLATFORM_MAP = {'AND': 'android', 'IOS': 'ios'}
actuals = pd.DataFrame()
if refresh_data:
    
    actuals = bqc.get('./sql/actuals.sql', is_path=True, query_parameters=params)
    pd.to_pickle(actuals, './data/actuals.pkl')
else:
    actuals = pd.read_pickle('./data/actuals.pkl')

actuals['dt'] = pd.to_datetime(actuals['dt'])
actuals['platform'] = actuals['platform'].map(PLATFORM_MAP).fillna(actuals['platform'].str.lower())
actuals = actuals.sort_values('dt')

# Anchor DAU: last observed day per platform
anchor_dau = actuals.sort_values('dt').groupby('platform')['dau'].last().to_dict()

In [5]:
#cost = bqc.print_cost_estimate('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
#cost = bqc.print_cost_estimate('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)

In [6]:
params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}

if refresh_data:
    live_retention  = bqc.get('./sql/retention.sql',  is_path=True, query_parameters=params)
    live_retention.to_pickle('./data/live_retention.pkl')
    live_conversion = bqc.get('./sql/conversion.sql', is_path=True, query_parameters=params)
    live_conversion.to_pickle('./data/live_conversion.pkl')
else:
    live_retention  = pd.read_pickle('./data/live_retention.pkl')
    live_conversion = pd.read_pickle('./data/live_conversion.pkl')

live_retention['platform']  = live_retention['platform'].map(PLATFORM_MAP).fillna(live_retention['platform'].str.lower())
live_conversion['platform'] = live_conversion['platform'].map(PLATFORM_MAP).fillna(live_conversion['platform'].str.lower())

In [7]:
params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}

if refresh_data:
    marketing  = bqc.get('./sql/marketing.sql',  is_path=True, query_parameters=params)
    marketing.to_pickle('./data/marketing.pkl')
else:
    marketing  = pd.read_pickle('./data/marketing.pkl')


In [8]:
# CPI and UA spend are loaded on demand via "Get from actuals" buttons in the panel

In [9]:
# show
from common_lib.app import prefill_panel, setup_callbacks

engine = SimulationEngine()
panel  = ScenarioPanel(saved_scenarios=list_scenarios(), last_actuals_date=actuals['dt'].dt.date.max())
prefill_panel(panel, actuals, anchor_dau)
setup_callbacks(panel, engine, actuals,
                live_retention=live_retention, live_conversion=live_conversion,
                installs=actuals, marketing=marketing)
panel.display()

<IPython.core.display.Javascript object>

In [10]:
from common_lib.plots import plot, plot_retention, plot_conversion, configure as configure_plots
from common_lib.simulation import list_results

configure_plots(actuals)

# ── Usage ──────────────────────────────────────────────────────────────────
# plot('base_case')                         # all charts
# plot('base_case', chart='dau')            # DAU only
# plot('base_case', chart='revenue')        # daily revenue
# plot('base_case', chart='monthly')        # monthly bar
# plot(['base_case', 'high_ua'])            # compare scenarios
#
# plot_retention('base_case')               # retention curve from saved scenario
# plot_conversion('base_case')              # conversion curve from saved scenario
# plot_retention(['base_case', 'high_ua'])  # compare retention curves across scenarios
# plot_retention(panel.get_curve_anchors()) # preview current panel state
#print('Available results:', list_results())

In [11]:
#plot('test3', chart='dau')

In [12]:
#plot('test3', chart='revenue')

In [13]:
styled, results = comparison_table(actuals=actuals)
styled


,DAU 2027-03,Cumul Margin 2027-03,vs Target 2027-03,DAU 2027-12,Cumul Margin 2027-12,vs Target 2027-12,DAU 2029-12,Cumul Margin 2029-12,vs Target 2029-12
Scenario,,,,,,,,,
"plan a - 200k 202607, prd uplifts, org uplifts","34,897","$11,242,378",✓ +6.1%,"18,250","$13,900,822",—,"8,589","$17,314,113",—
"plan a - 200k 202607, prd uplifts","31,694","$10,901,364",✓ +2.9%,"17,545","$13,401,500",—,"8,448","$16,722,069",—
"plan a - 200k 202607 50k 2x, prd uplifts, org uplifts","35,322","$11,233,829",✓ +6.1%,"18,377","$13,914,904",—,"8,615","$17,344,907",—
"plan a - 200k 202607 50k x2, prd uplifts","32,119","$10,892,814",✓ +2.8%,"17,672","$13,415,583",—,"8,473","$16,752,862",—
"plan a - 200k 2026decr, prd uplifts, org uplifts","38,026","$11,157,269",✓ +5.3%,"19,046","$13,975,340",—,"8,747","$17,492,529",—
"plan a - 200k 2026decr, prd uplifts","34,823","$10,816,254",✓ +2.1%,"18,341","$13,476,019",—,"8,605","$16,900,485",—
"plan a - no ua, no uplifts","31,644","$10,531,431",✗ -0.6%,"17,528","$12,819,124",—,"8,444","$15,859,212",—


In [14]:
#results.sort_values('Cumul Margin 2027-12', ascending=False)

In [15]:
#export_all_tables(actuals=actuals)

In [16]:
## Things to do next
#- Split IAP from IAA revenue
#- Add an export all / run all function
#- Review and reformulate organic boost uplift
#    - Why is there new installs showing in lesser number that organic uplift
#    - Rethink where the percentage should be applied to
#- 